# 4. Multiclass Classification Models

This notebook explores experiments with combined classes:
- **Experiment 1**: Normal vs Any Tumor (0 vs 1+2+3) - most inclusive tumor detection
- **Experiment 4**: Normal vs Actual Tumor (0 vs 2+3) - excluding normal-from-tumor

All models are evaluated on both validation and held-out test sets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/new_work/Projects/Camelyon16/camelyon16-pathology

!pip -q install -r requirements.txt

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

np.random.seed(42)
tf.random.set_seed(42)

TRAIN_DATASET_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_4class_stain_normalised'
TEST_DATASET_PATH = '/content/drive/MyDrive/new_work/Projects/pathovis_project/data/camelyon16_test_stain_normalised'

## 4.1 Experiment 1: Normal vs Any Tumor

**Question**: Can the model detect ANY tumor-related tissue?

This is the most inclusive definition of "tumor" - includes:
- Normal tissue from tumor slides (class 1)
- Boundary regions (class 2)
- Pure tumor (class 3)

Class mapping:
- 0: normal_from_normal
- 1: normal_from_tumor + boundary_tumor + pure_tumor

In [ ]:
from src.models import run_binary_experiment
from config import DEFAULT_CONFIG

DEFAULT_CONFIG.training.val_max_samples_per_class = 4000
DEFAULT_CONFIG.training.normalise_patches = True

print("=" * 60)
print("EXPERIMENT 1: Normal vs Any Tumor")
print("=" * 60)

exp1_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=1,  # Normal vs Any Tumor (0 vs 1,2,3)
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

exp1_model = exp1_results['model']
print(f"\nValidation Results:")
print(f"  Accuracy: {exp1_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp1_results['results']['auc']:.3f}")

## 4.2 Experiment 4: Normal vs Actual Tumor

**Question**: Can the model detect actual tumor tissue (excluding normal-from-tumor)?

This is a stricter definition - only tissue with tumor cells:
- Boundary regions (class 2)
- Pure tumor (class 3)

Class mapping:
- 0: normal_from_normal
- 1: boundary_tumor + pure_tumor

In [ ]:
DEFAULT_CONFIG.training.normalise_patches = True

print("=" * 60)
print("EXPERIMENT 4: Normal vs Actual Tumor")
print("=" * 60)

exp4_results = run_binary_experiment(
    dataset_path=TRAIN_DATASET_PATH,
    experiment_type=4,  # Normal vs Actual Tumor (0 vs 2,3)
    model_name='subtle',
    epochs=15,
    learning_rate=1e-5
)

exp4_model = exp4_results['model']
print(f"\nValidation Results:")
print(f"  Accuracy: {exp4_results['results']['accuracy']:.1%}")
print(f"  AUC: {exp4_results['results']['auc']:.3f}")

## 4.3 Test Set Evaluation

Evaluate both experiments on the held-out test set.

In [ ]:
from src.models import evaluate_on_test_set

# Extract optimal thresholds from validation results
exp1_threshold = exp1_results['results']['threshold']
exp4_threshold = exp4_results['results']['threshold']

# Experiment 1 test evaluation
print("=" * 60)
print("TEST SET EVALUATION: Experiment 1 - Normal vs Any Tumor")
print("=" * 60)
exp1_class_mapping = {0: ['normal_from_normal'], 1: ['normal_from_tumor', 'boundary_tumor', 'pure_tumor']}
exp1_test_results = evaluate_on_test_set(exp1_model, TEST_DATASET_PATH, exp1_class_mapping, "exp1", threshold=exp1_threshold)
print(f"Threshold (from val): {exp1_threshold:.3f}")
print(f"Accuracy: {exp1_test_results['accuracy']:.1%}")
print(f"AUC: {exp1_test_results['auc']:.3f}")
print(exp1_test_results['report'])

# Experiment 4 test evaluation
print("=" * 60)
print("TEST SET EVALUATION: Experiment 4 - Normal vs Actual Tumor")
print("=" * 60)
exp4_class_mapping = {0: ['normal_from_normal'], 1: ['boundary_tumor', 'pure_tumor']}
exp4_test_results = evaluate_on_test_set(exp4_model, TEST_DATASET_PATH, exp4_class_mapping, "exp4", threshold=exp4_threshold)
print(f"Threshold (from val): {exp4_threshold:.3f}")
print(f"Accuracy: {exp4_test_results['accuracy']:.1%}")
print(f"AUC: {exp4_test_results['auc']:.3f}")
print(exp4_test_results['report'])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

experiments = ['Exp 1: Normal vs\nAny Tumor', 'Exp 4: Normal vs\nActual Tumor']

val_accs = [exp1_results['results']['accuracy'], exp4_results['results']['accuracy']]
test_accs = [exp1_test_results['accuracy'], exp4_test_results['accuracy']]
val_aucs = [exp1_results['results']['auc'], exp4_results['results']['auc']]
test_aucs = [exp1_test_results['auc'], exp4_test_results['auc']]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(experiments))
width = 0.35

# Accuracy comparison
axes[0].bar(x - width/2, val_accs, width, label='Validation', color='steelblue')
axes[0].bar(x + width/2, test_accs, width, label='Test', color='darkorange')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Validation vs Test Accuracy')
axes[0].set_xticks(x)
axes[0].set_xticklabels(experiments)
axes[0].legend()
axes[0].set_ylim(0.5, 1.0)
for i, (v, t) in enumerate(zip(val_accs, test_accs)):
    axes[0].text(i - width/2, v + 0.02, f'{v:.1%}', ha='center', fontsize=9)
    axes[0].text(i + width/2, t + 0.02, f'{t:.1%}', ha='center', fontsize=9)

# AUC comparison
axes[1].bar(x - width/2, val_aucs, width, label='Validation', color='steelblue')
axes[1].bar(x + width/2, test_aucs, width, label='Test', color='darkorange')
axes[1].set_ylabel('AUC')
axes[1].set_title('Validation vs Test AUC')
axes[1].set_xticks(x)
axes[1].set_xticklabels(experiments)
axes[1].legend()
axes[1].set_ylim(0.5, 1.0)
for i, (v, t) in enumerate(zip(val_aucs, test_aucs)):
    axes[1].text(i - width/2, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
    axes[1].text(i + width/2, t + 0.02, f'{t:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('exp1_exp4_val_vs_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.4 Future: True 4-Class Classification

*To be implemented*

Train a single model to classify all 4 classes simultaneously.